# W8 — Variance Swap Hedging & VRP Backtest
Group 4 | Person 1

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy import stats

from src.pricing.vrp_backtest import (
    demonstrate_model_free_hedge,
    simulate_daily_pnl,
    run_vrp_backtest,
    HESTON_PARAMS, HESTON_PHYSICAL,
    N_MONTHS_BACKTEST, VEGA_TARGET, MONTHS_PER_YEAR,
    plot_model_free_hedge, plot_daily_pnl, plot_backtest,
)


## Part A — Model-Free Vega Hedge

Hedge weights: $w(K) = \\frac{2 e^{rT} \\Delta K}{T K^2}$

Key point: weights depend only on the strike grid — no model parameters involved.

In [ ]:
hedge_data = demonstrate_model_free_hedge()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# left: hedge weight curve
strikes = hedge_data['strikes']
weights = hedge_data['weights']
axes[0].plot(strikes, weights * 1e4, color='steelblue', lw=2.5)
axes[0].fill_between(strikes, weights * 1e4, alpha=0.2, color='steelblue')
axes[0].axvline(100, color='grey', ls='--', lw=1, label='ATM')
axes[0].set_title('w(K) = 2*exp(rT)*dK / (T*K^2)')
axes[0].set_xlabel('Strike K')
axes[0].set_ylabel('Weight x 10^4')
axes[0].legend()
axes[0].grid(alpha=0.3)

# right: same weights -> different K_var for different IV surfaces
vals = [hedge_data['model_free_var_flat_surface']*10000,
        hedge_data['model_free_var_skewed_surface']*10000]
bars = axes[1].bar(['Flat IV (20%)', 'Skewed IV'], vals,
                   color=['steelblue', 'mediumpurple'], alpha=0.8, width=0.4)
for b, v in zip(bars, vals):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.2, f'{v:.1f}', ha='center')
axes[1].set_title('Same Weights, Different IV Surfaces')
axes[1].set_ylabel('Implied Variance (x10^4)')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/w8/w8_model_free_hedge.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Flat IV -> var = {vals[0]:.2f}")
print(f"Skewed IV -> var = {vals[1]:.2f}")
print("Weights are the same in both cases -- model-free confirmed")


## Part B — Daily Vega P&L: Hedged vs Unhedged

In [ ]:
pnl_data = simulate_daily_pnl(HESTON_PARAMS, n_paths=500, seed=77)

days = range(len(pnl_data['mean_pnl_unhedged']))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(days, pnl_data['mean_pnl_unhedged'], color='tomato', lw=2, label='Unhedged')
axes[0].plot(days, pnl_data['mean_pnl_hedged'], color='seagreen', lw=2, label='Hedged')
axes[0].axhline(0, color='grey', ls='--', lw=0.8)
axes[0].set_title('Cumulative Vega P&L')
axes[0].set_xlabel('Trading Day')
axes[0].set_ylabel('P&L ($)')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda x,_: f'${x:,.0f}'))

axes[1].hist(pnl_data['all_terminal_unhedged'], bins=35, color='tomato', alpha=0.6, density=True,
             label=f"Unhedged  sigma=${pnl_data['std_terminal_unhedged']:,.0f}")
axes[1].hist(pnl_data['all_terminal_hedged'], bins=35, color='seagreen', alpha=0.6, density=True,
             label=f"Hedged    sigma=${pnl_data['std_terminal_hedged']:,.0f}")
axes[1].set_title(f"Terminal P&L  |  Var reduction: {pnl_data['var_reduction_pct']:.1f}%")
axes[1].set_xlabel('Terminal P&L ($)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/w8/w8_daily_pnl.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Notional: ${pnl_data['notional']:,.0f}")
print(f"K_var:    {pnl_data['k_var']*10000:.1f} bps")
print(f"Unhedged sigma: ${pnl_data['std_terminal_unhedged']:,.0f}")
print(f"Hedged sigma:   ${pnl_data['std_terminal_hedged']:,.0f}")


## Part C — VRP Backtest 2019–2024

Monthly P&L = $K_{var} - RV - $ transaction costs. Position sized so initial Vega = \$10,000.

In [ ]:
result = run_vrp_backtest()

df = result.monthly_pnl
eq = result.equity_curve

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# equity curve
axes[0].plot(eq.index, eq.values, color='steelblue', lw=2.5)
axes[0].fill_between(eq.index, eq.values, 100, alpha=0.15, color='steelblue')
axes[0].axhline(100, color='grey', ls='--', lw=0.7)
axes[0].set_title('Equity Curve — Short Variance Swap (2019–2024)')
axes[0].set_ylabel('NAV (base=100)')
axes[0].grid(alpha=0.25)

# monthly P&L bars
bar_c = ['seagreen' if v > 0 else 'tomato' for v in df['net_pnl']]
axes[1].bar(df.index, df['net_pnl'], color=bar_c, alpha=0.85, width=20)
axes[1].axhline(0, color='grey', lw=0.8)
axes[1].set_title('Monthly Net P&L (K_var - RV - tc)')
axes[1].set_ylabel('P&L ($)')
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.savefig('../outputs/w8/w8_backtest.png', dpi=150, bbox_inches='tight')
plt.show()


## Monthly P&L Table

In [ ]:
tbl = df[['k_var','rv','vrp_vol','net_pnl']].copy()
tbl.columns = ['K_var','RV','VRP (vol pts)','Net P&L ($)']
tbl['Net P&L ($)'] = tbl['Net P&L ($)'].map('${:,.0f}'.format)
tbl['VRP (vol pts)'] = tbl['VRP (vol pts)'].map('{:.2f}'.format)
tbl


In [ ]:
# basic summary stats for the backtest
r = df['net_pnl'] / VEGA_TARGET
sharpe  = r.mean() / r.std() * (MONTHS_PER_YEAR ** 0.5)
cum     = (1 + r).cumprod()
max_dd  = float(((cum - cum.cummax()) / cum.cummax()).min())
win_rate = (r > 0).mean() * 100

print(f"Sharpe:       {sharpe:.2f}")
print(f"Max Drawdown: {max_dd*100:.1f}%")
print(f"Win Rate:     {win_rate:.0f}%")


# W8 — Beta Decomposition, VRP Regimes, and Metrics


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.analysis.vrp_metrics import run_metrics_analysis


In [ ]:
df_metrics, betas, reg_summary, metrics_table, regime_df = run_metrics_analysis()


In [ ]:
betas


In [ ]:
reg_summary


In [ ]:
metrics_table


In [ ]:
regime_df[['vix', 'k_var', 'rv', 'vrp_vol', 'net_pnl', 'regime']].head(15)
